# Muhr — Paper 1 tajribalari (E3, E4, E5)

**Bir martalik tayyorgarlik (Add Input):**
1. `muhr-v2` — shu bloknot bilan berilgan `Muhr_v2.zip` ni **Dataset** sifatida yuklang (ichida `muhr_proj/`).
2. `best.pt` turgan dataset yoki eski bloknot chiqishi (`runs/taklif/best.pt`).
3. COCO 2017 dataseti (`awsaf49/coco-2017-dataset`: `train2017`, `val2017`, `test2017`).
4. Faqat `e3_div2k` uchun: DIV2K dataseti (`DIV2K_train_HR`, `DIV2K_valid_HR` papkalari bilan).

**Har bir ishga tushirish:**
1. Pastdagi `STAGE` ni tanlang.
2. **Accelerator: GPU T4 (yoki P100)**, **Internet: On** (LPIPS og'irliklari uchun).
3. **Save Version → Save & Run All (Commit)**. Natija Output'da `<STAGE>.zip` bo'lib saqlanadi.
4. Agar 11 soatga sig'masa, shu versiya chiqishini **Add Input** qilib, xuddi shu `STAGE` bilan qayta Commit qiling: tayyor bo'laklar qayta hisoblanmaydi.

| STAGE | Tajriba | Taxminiy hajm |
|---|---|---|
| `e4_unmarked` | E4: 10 000 belgisiz COCO test2017 + 500 degenerat tasvir, 5 ta zararsiz o'zgarish, barcha gipotezalar | birinchi |
| `e3_coco` | E3/E-OP: 2 000 yangi COCO tasviri (val2017[1500:3500]), A va G kodeklar, global rejim, 27 hujum | ikkinchi |
| `e3_coco_tiled` | E3/H4/E6: shu tasvirlarning 500 tasi, bloklar rejimi | uchinchi |
| `e6_stn_off` | Ablyatsiya qayta o'qitishsiz: 500 tasvir, G, STN sinov vaqtida o'chirilgan | ≈ 0.7 GPU-soat |
| `e3_div2k` | E3: DIV2K 500 ta tasvir (1024 px gacha), A va G | ≈ 1.5 GPU-soat |
| `e8_games` | E8: nusxa / kollaj / o'rtacha / qayta yozish o'yinlari, 300 tasvir | ≈ 0.5–1 GPU-soat [taxmin] |

**v2.3 (2026-11-13):** E5 o'qitish bosqichlari olib tashlandi (qayta o'qitish ablyatsiyasi kvotaga sig'maydi).

**v2.1 (2026-11-13):** qidiruv chegarasi `--max-n 4` (COCO uchun; DIV2K uchun 8), fp16 o'z-o'zini tekshirish bilan, paketli dekodlash. Log boshida `AMP check: {...}` qatori chiqadi.

Birinchi ishga tushirishdan keyin log'dagi `chunk ... saved (..., N s)` qatorini menga yuboring — GPU vaqtini shundan hisoblaymiz.


In [ ]:
# ===== Sozlamalar =====
STAGE = "e8_games"      # e8_games | e3_div2k   (bajarilgan: e4_unmarked, e3_coco, e3_coco_tiled, e6_stn_off)
MAX_HOURS = 11.0        # Kaggle sessiyasi 12 soat; 1 soat zaxira


In [ ]:
import glob, os, shutil, subprocess, sys, time
from pathlib import Path
W, IN, TMP = Path("/kaggle/working"), Path("/kaggle/input"), Path("/kaggle/tmp")
TMP.mkdir(parents=True, exist_ok=True)

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"buyruq xato bilan tugadi: {cmd[:3]}")

def find_dir(name):
    hits = sorted(glob.glob(str(IN / "**" / name), recursive=True))
    hits = [h for h in hits if os.path.isdir(h)]
    return Path(hits[0]) if hits else None

# --- kod
def _ver(d):
    try:
        return tuple(int(x) for x in (d / "VERSION").read_text().strip().split("."))
    except Exception:
        return (0,)
src = sorted({Path(p).parents[1] for p in glob.glob(str(IN / "**/muhr_proj/muhr_exp/codec2.py"), recursive=True)},
             key=_ver, reverse=True)       # Input'da bir nechta nusxa bo'lsa, eng yangi versiya olinadi
assert src, "muhr-v2 dataseti (muhr_proj/muhr_exp) topilmadi — Add Input qiling"
print("kod:", src[0], "versiya", ".".join(map(str, _ver(src[0]))))
assert _ver(src[0]) >= (2, 5), "muhr-v2 eski versiyada — Input'da eng oxirgi versiyani (2.5) tanlang"
CODE = W / "muhr_proj"
if CODE.exists():
    shutil.rmtree(CODE)                     # avvalgi ishdan qolgan eski nusxani ishlatmaslik uchun
shutil.copytree(src[0], CODE)
run([sys.executable, "-m", "pip", "install", "-q", "galois", "lpips", "bchlib"])

# --- model
ck = sorted(glob.glob(str(IN / "**/runs/taklif/best.pt"), recursive=True))
assert ck, "runs/taklif/best.pt topilmadi — uni o'z ichiga olgan datasetni Add Input qiling"
CKPT = ck[0]; print("checkpoint:", CKPT)


In [ ]:
# --- ma'lumotlar (o'qitishda ishlatilmagan qismlar)
def link_subset(files, dst):
    dst = Path(dst)
    if dst.exists():
        return dst
    dst.mkdir(parents=True)
    for f in files:
        os.symlink(f, dst / Path(f).name)
    return dst

val = find_dir("val2017")
assert val is not None, "COCO val2017 topilmadi"
vfiles = sorted(glob.glob(str(val / "*.jpg")))
print("val2017:", len(vfiles))
# dissertatsiyada: [0:500] validatsiya, [500:1500] eski test. Yangi test: [1500:3500]
T_COCO = link_subset(vfiles[1500:3500], TMP / "test_coco")
VAL500 = link_subset(vfiles[:500], TMP / "val500")
TEST2017 = find_dir("test2017")
TRAIN2017 = find_dir("train2017")
def div2k_dir():
    """DIV2K yuqori aniqlikdagi (HR) PNG'lar: papka tuzilishi datasetga qarab har xil (ichma-ich papkalar ham),
    shuning uchun Input ichidagi barcha DIV2K PNG'lari rekursiv qidiriladi; LR (x2/x3/x4) nusxalar chiqarib tashlanadi."""
    files = [f for f in glob.glob(str(IN / "**" / "*.png"), recursive=True) if "div2k" in f.lower()]
    files = [f for f in files if not any(t in f for t in ("LR", "/X2", "/X3", "/X4", "x2.png", "x3.png", "x4.png"))]
    hr = sorted({Path(f).name: f for f in files}.values(), key=lambda f: Path(f).name)   # nomi bo'yicha takrorsiz
    print("DIV2K HR PNG:", len(hr), "| namuna:", hr[:2])
    if not hr:
        return None
    dst = TMP / "div2k"
    if dst.exists():
        shutil.rmtree(dst)
    return link_subset(hr, dst)
print("test2017:", TEST2017, "| train2017:", TRAIN2017)


In [ ]:
# --- oldingi Commit chiqishidan davom ettirish
OUT = W / STAGE
OUT.mkdir(exist_ok=True)
prev = [p for p in glob.glob(str(IN / "**" / f"{STAGE}.zip"), recursive=True)]
if prev and not any(OUT.iterdir()):
    print("oldingi natija topildi:", prev[0])
    shutil.unpack_archive(prev[0], OUT)
print("mavjud bo'laklar:", len(list(OUT.glob("chunk_*.npz"))))


In [ ]:
os.chdir(CODE)
py = sys.executable
if STAGE == "e4_unmarked":
    assert TEST2017 is not None, "COCO test2017 topilmadi"
    run([py, "scripts/e4_unmarked.py", "--ckpt", CKPT, "--data", TEST2017, "--n", 10000,
         "--degenerate", 500, "--out", OUT, "--max-hours", MAX_HOURS])
elif STAGE == "e3_coco":
    run([py, "scripts/e3_marked.py", "--ckpt", CKPT, "--data", T_COCO, "--n", 2000,
         "--designs", "A,G", "--modes", "global", "--out", OUT, "--max-hours", MAX_HOURS])
elif STAGE == "e3_coco_tiled":
    run([py, "scripts/e3_marked.py", "--ckpt", CKPT, "--data", T_COCO, "--n", 500,
         "--designs", "A", "--modes", "tiled", "--out", OUT, "--max-hours", MAX_HOURS])
elif STAGE == "e3_div2k":
    d = div2k_dir(); assert d is not None, "DIV2K topilmadi"
    run([py, "scripts/e3_marked.py", "--ckpt", CKPT, "--data", d, "--n", 500, "--max-side", 1024,
         "--designs", "A,G", "--modes", "global", "--chunk", 25, "--out", OUT, "--max-hours", MAX_HOURS])
elif STAGE == "e6_stn_off":      # qayta o'qitishsiz ablyatsiya: o'qitilgan dekoderda STN sinov vaqtida o'chiriladi
    run([py, "scripts/e3_marked.py", "--ckpt", CKPT, "--data", T_COCO, "--n", 500, "--designs", "G",
         "--modes", "global", "--no-stn-infer", "--out", OUT, "--max-hours", MAX_HOURS])
elif STAGE == "e8_games":        # E8 xavfsizlik o'yinlari: nusxa, kollaj, o'rtacha, qayta yozish (300 qurbon + 300 xost)
    run([py, "scripts/e8_games.py", "--ckpt", CKPT, "--data", T_COCO, "--n", 300, "--design", "G9",
         "--out", OUT, "--max-hours", MAX_HOURS])
else:
    raise SystemExit("noma'lum STAGE")


In [ ]:
# --- natijani bitta zip qilib saqlash (Output'da ko'rinadi)
os.chdir(W)
z = shutil.make_archive(str(W / STAGE), "zip", OUT)
print("saqlandi:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
print("bo'laklar:", len(list(OUT.glob("chunk_*.npz"))))
